In [ ]:
import re
from pathlib import Path
import numpy as np
from scipy.spatial.transform import Rotation
from typing import List, Dict
import plotly.graph_objects as go
world_path = "C:/Users/andre/OneDrive/Documents/EPFL-DESKTOP-0FFTIDB/Xplore/DISAL_Drone_Project/epfl_code/worlds/crazyflie_world_assignment.wbt"

In [ ]:
def extract_gate_data(content: str) -> List[Dict]:
    """Extract complete gate data from VRML content, including beam subfields."""
    gate_pattern = re.compile(
        r'DEF (\w+)\s+RacingGate\s*{(.*?)}',
        re.DOTALL
    )

    beam_fields = [
        'topBeamTranslation', 'topBeamScale',
        'bottomBeamTranslation', 'bottomBeamScale',
        'leftBeamTranslation', 'leftBeamScale',
        'rightBeamTranslation', 'rightBeamScale',
        'leftLegTranslation', 'rightLegTranslation'
    ]

    gates = []

    for match in gate_pattern.finditer(content):
        gate_name = match.group(1)
        block = match.group(2)

        gate = {
            'type': 'gate',
            'name': gate_name,
            'raw_data': match.group(0)
        }

        # Extract basic fields
        translation_match = re.search(r'translation\s+([\d\.\-eE]+)\s+([\d\.\-eE]+)\s+([\d\.\-eE]+)', block)
        rotation_match = re.search(r'rotation\s+([\d\.\-eE]+)\s+([\d\.\-eE]+)\s+([\d\.\-eE]+)\s+([\d\.\-eE]+)', block)
        goal_size_match = re.search(r'goalSize\s+([\d\.\-eE]+)\s+([\d\.\-eE]+)\s+([\d\.\-eE]+)', block)

        if translation_match:
            gate['translation'] = list(map(float, translation_match.groups()))
        if rotation_match:
            gate['rotation'] = list(map(float, rotation_match.groups()))
        if goal_size_match:
            gate['scale'] = list(map(float, goal_size_match.groups()))

        # Extract all beam fields
        for key in beam_fields:
            match = re.search(fr'{key}\s+([\d\.\-eE]+)\s+([\d\.\-eE]+)\s+([\d\.\-eE]+)', block)
            if match:
                gate[key] = list(map(float, match.groups()))

        gates.append(gate)

    return gates

def extract_takeoff_pad(content: str) -> List[Dict]:
    """Extract raw takeoff pad data from VRML content"""

    takeoff_pad_pattern = re.compile(
        r'DEF\s+TAKE_OFF_PAD\s+Solid\s*{[^}]*?'
        r'translation\s+([\d\.\-eE]+)\s+([\d\.\-eE]+)\s+([\d\.\-eE]+)\s*'
        r'rotation\s+([\d\.\-eE]+)\s+([\d\.\-eE]+)\s+([\d\.\-eE]+)\s+([\d\.\-eE]+).*?'
        r'geometry\s+Box\s*{\s*size\s+([\d\.\-eE]+)\s+([\d\.\-eE]+)\s+([\d\.\-eE]+)',
        re.DOTALL
    )

    return [
        {
            'type': 'takeoff_pad',
            'name': 'TAKE_OFF_PAD',
            'translation': list(map(float, match.groups()[0:3])),
            'rotation': list(map(float, match.groups()[3:7])),
            'scale': list(map(float, match.groups()[7:10])),
            'raw_data': match.group(0)
        }
        for match in takeoff_pad_pattern.finditer(content)
    ]

def extract_beams(content: str) -> List[Dict]:
    """Extract beam data from RacingGate blocks"""

    gate_pattern = re.compile(
        r'DEF (\w+)\s+RacingGate\s*{.*?}',
        re.DOTALL
    )

    beam_names = [
        ('topBeam', True),
        ('bottomBeam', True),
        ('leftBeam', True),
        ('rightBeam', True),
    ]

    beams = []

    for match in gate_pattern.finditer(content):
        block = match.group(0)

        # Get gate-level translation and rotation
        gate_translation = list(map(float, re.search(r'translation\s+([\d\.\-eE]+)\s+([\d\.\-eE]+)\s+([\d\.\-eE]+)', block).groups()))
        gate_rotation = list(map(float, re.search(r'rotation\s+([\d\.\-eE]+)\s+([\d\.\-eE]+)\s+([\d\.\-eE]+)\s+([\d\.\-eE]+)', block).groups()))
        gate_rot = Rotation.from_rotvec(np.array(gate_rotation[:3]) * gate_rotation[3])

        for name, has_scale in beam_names:
            trans_match = re.search(fr'{name}Translation\s+([\d\.\-eE]+)\s+([\d\.\-eE]+)\s+([\d\.\-eE]+)', block)
            scale_match = re.search(fr'{name}Scale\s+([\d\.\-eE]+)\s+([\d\.\-eE]+)\s+([\d\.\-eE]+)', block)

            if trans_match and scale_match:
                local_translation = np.array(list(map(float, trans_match.groups())))
                scale = list(map(float, scale_match.groups()))

                # Transform beam local position to world position
                world_translation = gate_rot.apply(local_translation) + gate_translation

                beams.append({
                    'type': 'beam',
                    'name': f'{name}_{match.group(1)}',
                    'translation': world_translation.tolist(),
                    'rotation': gate_rotation,
                    'scale': scale,
                    'raw_data': block
                })

    return beams


def extract_object_data(world_path: Path) -> List[Dict]:
    """Main extraction function for gates and takeoff pads"""
    with open(world_path, 'r') as f:
        content = f.read()

    gate_data = extract_gate_data(content)
    takeoff_pad_data = extract_takeoff_pad(content)
    # beams_data = extract_beams(content)

    return gate_data + takeoff_pad_data

In [ ]:
def transform_coordinates(objects):
    """Apply coordinate system transformations and handle gate+beam expansion."""
    transformed = []

    for obj in objects:
        if obj['type'] == 'gate':
            transformed.extend(transform_gate_and_beams(obj))
        else:
            transformed.append(transform_generic_object(obj))

    return transformed


def transform_gate_and_beams(gate_obj):
    """Transform gate and its beams to the world frame and coordinate system."""
    transformed = []

    # Transform the gate itself
    gate_transformed = transform_generic_object(gate_obj)
    transformed.append(gate_transformed)

    # Extract and transform beams
    beams = convert_gates_to_beam_objects([gate_obj])
    for beam in beams:
        transformed_beam = transform_generic_object(beam)
        transformed.append(transformed_beam)

    return transformed

def transform_generic_object(obj):
    """Coordinate system transformation for a single object."""

    x, y, z = obj['translation']
    rx, ry, rz, angle = obj['rotation']

    # 1. Apply offset
    new_x = x + 1
    new_y = y + 1
    new_z = z

    # If it's a gate, apply a clean rotation rewrite
    if obj['type'] == 'gate':
        new_rotation = [0, 0, 1, angle]
    else:
        # Leave the rotation unchanged for beams
        new_rotation = [rx, ry, rz, angle]

    return {
        **obj,
        'translation': [new_x, new_y, new_z],
        'rotation': new_rotation,
        'original_translation': obj['translation']
    }



In [ ]:
def convert_gates_to_beam_objects(gates: List[Dict]) -> List[Dict]:
    """Extracts and transforms beam components from gate objects into world-space obstacles."""
    beam_objects = []

    for gate in gates:
        gate_pos = np.array(gate['translation'])
        gate_rotvec = np.array(gate['rotation'][:3]) * gate['rotation'][3]
        gate_rot = Rotation.from_rotvec(gate_rotvec)

        print(f"\n=== Processing {gate['name']} ===")
        print(f"  Gate Position: {gate_pos}")
        print(f"  Gate Rotation vec: {gate_rotvec} (angle={np.linalg.norm(gate_rotvec)})")

        beam_defs = [
            ('top_beam', 'topBeamTranslation', 'topBeamScale'),
            ('bottom_beam', 'bottomBeamTranslation', 'bottomBeamScale'),
            ('left_beam', 'leftBeamTranslation', 'leftBeamScale'),
            ('right_beam', 'rightBeamTranslation', 'rightBeamScale'),
        ]

        for beam_type, trans_key, scale_key in beam_defs:
            if trans_key in gate and scale_key in gate:
                local_pos = np.array(gate[trans_key])
                scale = np.array(gate[scale_key])

                world_pos, full_rotation = transform_relative_to_gate(
                    gate_pos, gate_rot, beam_type, local_pos, scale
                )

                beam_objects.append({
                    'type': 'beam',
                    'name': f"{gate['name']}_{beam_type}",
                    'translation': world_pos.tolist(),
                    'rotation': full_rotation,
                    'scale': scale.tolist()
                })
            else:
                print(f"  [SKIP] {beam_type}: Missing keys")

    return beam_objects



def transform_relative_to_gate(gate_pos, gate_rot, beam_name, local_pos, scale):
    """
    Transforms a beam's local position to world space using the gate's position and rotation.
    Applies beam-specific local rotations before applying the gate's rotation.
    """

    # Define beam-specific local rotations
    if 'top_beam' in beam_name or 'bottom_beam' in beam_name:
        local_rot = Rotation.from_euler('z', np.pi / 2)  # rotate to be parallel with gate
    elif 'left_beam' in beam_name or 'right_beam' in beam_name:
        local_rot = Rotation.from_euler('y', np.pi / 2)  # rotate to be coplanar with gate
    else:
        local_rot = Rotation.identity()

    # Apply local beam rotation first, then gate's rotation
    world_rot = gate_rot * local_rot

    # Apply gate rotation to translate position (only gate_rot, not local_rot!)
    world_pos = gate_rot.apply(local_pos) + gate_pos

    # Compose final rotation vector
    rotvec = world_rot.as_rotvec()
    angle = np.linalg.norm(rotvec)
    axis = (rotvec / angle).tolist() if angle > 1e-6 else [0, 0, 1]
    full_rotation = axis + [angle]

    print(f"\n    [{beam_name}]")
    print(f"      Local Translation: {local_pos}")
    print(f"      Local Scale: {scale}")
    print(f"      → World Position: {world_pos}")
    print(f"      → Final Rotation (axis + angle): {full_rotation}")

    return world_pos, full_rotation


In [ ]:
def create_occupancy_map(objects, world_size=(10, 10, 5), resolution=0.05):
    grid_dims = (np.array(world_size) / resolution).astype(int)
    print(f"Creating {world_size}m world grid: {grid_dims} cells")

    occupancy_grid = np.zeros(grid_dims, dtype=np.int8)

    # Mark flight area (-4)
    flight_cells = int(8/resolution)
    start = int(1/resolution)  # 2m offset
    occupancy_grid[start:start+flight_cells,
                  start:start+flight_cells,
                  0] = -4

    for obj in objects:
        pos = np.array(obj['translation'])
        size = np.array(obj['scale'])
        rot = Rotation.from_rotvec(obj['rotation'][3] * np.array(obj['rotation'][:3]))

        half_size = size / 2
        x_range = np.arange(
            max(0, pos[0]-half_size[0]),
            min(world_size[0], pos[0]+half_size[0]),
            resolution
        )
        y_range = np.arange(
            max(0, pos[1]-half_size[1]),
            min(world_size[1], pos[1]+half_size[1]),
            resolution
        )
        z_range = np.arange(
            max(0, pos[2]-half_size[2]),
            min(world_size[2], pos[2]+half_size[2]),
            resolution
        )

        xx, yy, zz = np.meshgrid(x_range, y_range, z_range, indexing='ij')
        points = np.column_stack((xx.ravel(), yy.ravel(), zz.ravel()))
        rotated_points = rot.apply(points - pos) + pos
        grid_coords = (rotated_points / resolution).astype(int)

        valid = np.all((grid_coords >= 0) & (grid_coords < grid_dims), axis=1)
        unique_coords = np.unique(grid_coords[valid], axis=0)

        # Assign based on object type
        if obj['type'] == 'gate':
            value = -1
        elif obj['type'] in ('takeoff_pad', 'beam'):
            value = 1
        else:
            value = 0
        if unique_coords.size > 0:
            occupancy_grid[
                unique_coords[:, 0],
                unique_coords[:, 1],
                unique_coords[:, 2]
            ] = value

    return occupancy_grid


def plot_occupancy_with_plotly(occupancy_grid, resolution):
    fig = go.Figure()

    # Flight area (-4)
    flight_coords = np.where(occupancy_grid == -4)
    if len(flight_coords[0]) > 0:
        fig.add_trace(go.Scatter3d(
            x=flight_coords[0] * resolution,
            y=flight_coords[1] * resolution,
            z=flight_coords[2] * resolution,
            mode='markers',
            marker=dict(size=1, color='gray', opacity=0.2),
            name='Flight Area (-4)'
        ))

    # Gates (-1)
    gate_coords = np.where(occupancy_grid == -1)
    if len(gate_coords[0]) > 0:
        stride = max(1, int(0.05 / resolution))
        fig.add_trace(go.Scatter3d(
            x=gate_coords[0][::stride] * resolution,
            y=gate_coords[1][::stride] * resolution,
            z=gate_coords[2][::stride] * resolution,
            mode='markers',
            marker=dict(size=2, color='red', opacity=0.6),
            name='Gates (-1)'
        ))

    # Takeoff Pad (1)
    pad_coords = np.where(occupancy_grid == 1)
    if len(pad_coords[0]) > 0:
        fig.add_trace(go.Scatter3d(
            x=pad_coords[0] * resolution,
            y=pad_coords[1] * resolution,
            z=pad_coords[2] * resolution,
            mode='markers',
            marker=dict(size=3, color='blue', opacity=0.7),
            name='Takeoff Pad (1)'
        ))

    # Flight Boundary Outline
    boundary_x = [1, 9, 9, 1, 1]
    boundary_y = [1, 1, 9, 9, 1]
    boundary_z = [0] * 5
    fig.add_trace(go.Scatter3d(
        x=boundary_x,
        y=boundary_y,
        z=boundary_z,
        mode='lines',
        line=dict(color='black', width=2),
        name='Flight Boundary'
    ))

    fig.update_layout(
        scene=dict(
            xaxis=dict(range=[0, 10], title='X (m)'),
            yaxis=dict(range=[0, 10], title='Y (m)'),
            zaxis=dict(range=[0, 5], title='Z (m)'),
            aspectmode='manual',
            aspectratio=dict(x=2, y=2, z=1)
        ),
        width=900,
        height=700,
        title="3D Occupancy Grid Map",
        showlegend=True,
    )

    fig.show()


In [ ]:
# 1. Extract and transform data
objects = extract_object_data(Path(world_path))
transformed_objects = transform_coordinates(objects)

# 2. Create occupancy map
resolution = 0.02 # 1 = meter resolution, 0.01 = centimeter resolution
occupancy_grid = create_occupancy_map(transformed_objects, resolution=resolution)

# 3. Visualize
plot_occupancy_with_plotly(occupancy_grid, resolution)
# InteractiveViewer(occupancy_grid, resolution=0.05)

# Save grid with all features
np.save("world_occupancy_map.npy", occupancy_grid)